In [1]:
!pip install fastapi uvicorn python-multipart pydicom opencv-python-headless torch torchvision --quiet
print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 19.3 MB/s eta 0:00:00
Dependencies installed.


In [4]:
!pip install pyngrok nest_asyncio --quiet
print("Done.")

Done.


In [5]:
import os
import io
import time
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import pydicom
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
import uvicorn
from pyngrok import ngrok
import nest_asyncio

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"All imports successful.")
print(f"Device: {device}")

All imports successful.
Device: cpu


In [6]:
TARGET_SIZE = 224

def preprocess_dicom_bytes(file_bytes: bytes) -> torch.Tensor:
    """
    Takes raw DICOM file bytes and returns a normalised tensor
    ready for model inference. Used by the FastAPI endpoint.
    """
    dcm = pydicom.dcmread(io.BytesIO(file_bytes))
    img = dcm.pixel_array.astype(np.float32)
    if len(img.shape) == 3:
        img = img[:, :, 0]
    img = cv2.resize(img, (TARGET_SIZE, TARGET_SIZE), interpolation=cv2.INTER_AREA)
    img_min, img_max = img.min(), img.max()
    if img_max > img_min:
        img = (img - img_min) / (img_max - img_min)
    else:
        img = np.zeros_like(img)
    tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    return tensor.to(device)


def load_model(model_path: str = None) -> nn.Module:
    """
    Loads the DenseNet-121 model. If a saved model path is provided
    it loads the trained weights. Otherwise uses random weights
    for demonstration purposes.
    """
    model = models.densenet121(weights=None)
    model.features.conv0 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    model.classifier = nn.Linear(model.classifier.in_features, 2)
    if model_path and os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))
        print(f"Trained weights loaded from {model_path}")
    else:
        print("No saved weights found. Using untrained model for demo.")
    model.eval()
    return model.to(device)


print("Preprocessing and model loading functions defined.")

Preprocessing and model loading functions defined.


In [7]:
# Load model
model = load_model()

# Build FastAPI app
app = FastAPI(title="GlobalMedNet Edge Node", version="1.0.0")

@app.get("/health")
def health_check():
    return {
        "status": "online",
        "model": "DenseNet-121",
        "version": "1.0.0",
        "device": str(device)
    }

@app.post("/analyse")
async def analyse_xray(file: UploadFile = File(...)):
    """
    Accepts a DICOM file upload and returns an AI diagnosis.
    This is the endpoint a hospital's PACS system would call.
    """
    start_time = time.time()

    # Read uploaded file
    file_bytes = await file.read()

    try:
        # Preprocess
        tensor = preprocess_dicom_bytes(file_bytes)

        # Run inference
        with torch.no_grad():
            output = model(tensor)
            probs  = F.softmax(output, dim=1)[0]
            pneumonia_confidence = float(probs[1])
            normal_confidence    = float(probs[0])
            prediction           = "Pneumonia detected" if pneumonia_confidence > 0.5 else "Normal"

        inference_time = (time.time() - start_time) * 1000

        return JSONResponse({
            "prediction":             prediction,
            "pneumonia_confidence":   round(pneumonia_confidence * 100, 2),
            "normal_confidence":      round(normal_confidence * 100, 2),
            "inference_time_ms":      round(inference_time, 1),
            "filename":               file.filename,
            "status":                 "success"
        })

    except Exception as e:
        return JSONResponse({"status": "error", "message": str(e)}, status_code=500)


@app.get("/model-info")
def model_info():
    return {
        "architecture":   "DenseNet-121",
        "input_size":     "224x224",
        "input_channels": 1,
        "output_classes": ["Normal", "Pneumonia"],
        "framework":      "PyTorch"
    }

print("FastAPI app defined.")
print("Endpoints:")
print("   GET  /health      - node health check")
print("   POST /analyse     - upload DICOM, get diagnosis")
print("   GET  /model-info  - model architecture details")

No saved weights found. Using untrained model for demo.
FastAPI app defined.
Endpoints:
   GET  /health      - node health check
   POST /analyse     - upload DICOM, get diagnosis
   GET  /model-info  - model architecture details


In [9]:
!ngrok authtoken 3BAL4KhqbDI8EaeZF6YgLjUZaoU_2VTUNBrnGgnrn4hriXS4n

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [11]:
import nest_asyncio
import threading
nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

import time
time.sleep(3)

ngrok_tunnel = ngrok.connect(8000)
public_url = ngrok_tunnel.public_url
print(f"GlobalMedNet Edge Node is live at: {public_url}")
print(f"API docs: {public_url}/docs")
print(f"Health check: {public_url}/health")

INFO:     Started server process [816]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


GlobalMedNet Edge Node is live at: https://overdiligent-ariel-filose.ngrok-free.dev
API docs: https://overdiligent-ariel-filose.ngrok-free.dev/docs
Health check: https://overdiligent-ariel-filose.ngrok-free.dev/health


In [12]:
import pydicom
import numpy as np
from pydicom.dataset import Dataset, FileDataset
from pydicom.uid import generate_uid
import datetime

def create_test_dicom(filename, label="test"):
    file_meta = pydicom.dataset.FileMetaDataset()
    file_meta.MediaStorageSOPClassUID = "1.2.840.10008.5.1.4.1.1.2"
    file_meta.MediaStorageSOPInstanceUID = generate_uid()
    file_meta.TransferSyntaxUID = pydicom.uid.ExplicitVRLittleEndian

    ds = FileDataset(filename, {}, file_meta=file_meta, preamble=b"\0" * 128)
    ds.PatientName = "Test^Patient"
    ds.PatientID = "123456"
    ds.Modality = "CR"
    ds.SOPClassUID = "1.2.840.10008.5.1.4.1.1.2"
    ds.SOPInstanceUID = generate_uid()
    ds.StudyDate = datetime.datetime.now().strftime("%Y%m%d")
    ds.Rows = 224
    ds.Columns = 224
    ds.BitsAllocated = 8
    ds.BitsStored = 8
    ds.HighBit = 7
    ds.PixelRepresentation = 0
    ds.SamplesPerPixel = 1
    ds.PhotometricInterpretation = "MONOCHROME2"

    # Simulate a chest X-ray pattern
    pixel_array = np.random.randint(30, 200, (224, 224), dtype=np.uint8)
    ds.PixelData = pixel_array.tobytes()
    ds.save_as(filename)
    print(f"Test DICOM created: {filename}")

create_test_dicom("test_xray.dcm")

from google.colab import files
files.download("test_xray.dcm")

Test DICOM created: test_xray.dcm


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
def create_pneumonia_test_dicom(filename):
    file_meta = pydicom.dataset.FileMetaDataset()
    file_meta.MediaStorageSOPClassUID = "1.2.840.10008.5.1.4.1.1.2"
    file_meta.MediaStorageSOPInstanceUID = generate_uid()
    file_meta.TransferSyntaxUID = pydicom.uid.ExplicitVRLittleEndian

    ds = FileDataset(filename, {}, file_meta=file_meta, preamble=b"\0" * 128)
    ds.PatientName = "Test^Patient"
    ds.PatientID = "789012"
    ds.Modality = "CR"
    ds.SOPClassUID = "1.2.840.10008.5.1.4.1.1.2"
    ds.SOPInstanceUID = generate_uid()
    ds.StudyDate = datetime.datetime.now().strftime("%Y%m%d")
    ds.Rows = 224
    ds.Columns = 224
    ds.BitsAllocated = 8
    ds.BitsStored = 8
    ds.HighBit = 7
    ds.PixelRepresentation = 0
    ds.SamplesPerPixel = 1
    ds.PhotometricInterpretation = "MONOCHROME2"

    pixel_array = np.random.randint(60, 220, (224, 224), dtype=np.uint8)
    pixel_array[80:140, 60:110]  = np.random.randint(180, 255, (60, 50), dtype=np.uint8)
    pixel_array[90:150, 120:170] = np.random.randint(170, 245, (60, 50), dtype=np.uint8)
    pixel_array[70:130, 150:200] = np.random.randint(160, 240, (60, 50), dtype=np.uint8)

    ds.PixelData = pixel_array.tobytes()
    ds.save_as(filename)
    print(f"Pneumonia test DICOM created: {filename}")

create_pneumonia_test_dicom("test_pneumonia.dcm")

from google.colab import files
files.download("test_pneumonia.dcm")

Pneumonia test DICOM created: test_pneumonia.dcm


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
from google.colab import files
uploaded = files.upload()

Saving best_model.pth to best_model.pth


In [16]:
model = load_model('best_model.pth')
print("Trained model loaded.")

Trained weights loaded from best_model.pth
Trained model loaded.


In [17]:
model = load_model('best_model.pth')
print("Trained model loaded.")

Trained weights loaded from best_model.pth
Trained model loaded.


In [18]:
import threading
import time

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(3)

ngrok_tunnel = ngrok.connect(8000)
public_url = ngrok_tunnel.public_url
print(f"Server live at: {public_url}")
print(f"API docs: {public_url}/docs")

INFO:     Started server process [816]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


Server live at: https://overdiligent-ariel-filose.ngrok-free.dev
API docs: https://overdiligent-ariel-filose.ngrok-free.dev/docs
